# Day 4: Colab 4D - From Field Sheet to DataFrame

Partners: your names here

Date: 09/03/2026

In [95]:
import pandas as pd
import numpy as np

url = 'https://eds-217-essential-python.github.io/data/messy_field_survey.csv'

survey = pd.read_csv(url)


df.drop_duplicates()                    # and subset=[...]

df.dropna(subset=['col'])               # remove rows missing a named column

df['col'] = df['col'].fillna(value)     # fill the gaps instead

df['col'] = df['col'].astype(float)     # int, float, str

df['col'] = df['col'].str.strip()       # .lower(), .replace(old, new)

df['new'] = expression                  # the derived-column pattern

def my_func(x):  ...                    # write it once, from this morning

df['new'] = df['col'].apply(my_func)    # run your own function down a column

df[df['col'] > value].copy()            # the filter pattern, from Day 3

df.rename(columns={'old': 'new'})       # renaming, from Day 2

### Part 1 : Find out what is wrong

In [96]:
survey.head()

,site,collection date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates
0,SITE_F,2025-07-28,24.1,5.89,5.51,899.1,4.0
1,site-d,2025-08-05,12.5,7.74,9.97,300.1,4.0
2,Site_C,2025-07-04,22.6,6.42,7.19,815.8,4.0
3,Site_D,2025-08-15,16.0,7.74,9.73,250.7,3.0
4,site_a,2025-06-24,14.7,"7,64",9.19,338.8,3.0


In [97]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   site                   320 non-null    object 
 1   collection date        320 non-null    object 
 2   temperature_c          302 non-null    float64
 3   pH                     320 non-null    object 
 4   dissolved_oxygen_mg_L  309 non-null    float64
 5   conductivity_uS_cm     313 non-null    float64
 6   n_replicates           315 non-null    float64
dtypes: float64(4), object(3)
memory usage: 17.6+ KB


In [98]:
survey.shape

(320, 7)

How many rows and columns? Run .head() and .info(). Which columns came in as object when you expected a number, and which came in as float64 when you expected a whole number?

    - there are 320 rows and 7 columns. the two columns that are objects are site and collection date. the 4 columns that came back as float64 are temperature_c, dissolved_oxygen_mg_L, conductivity_uS_cm, and n_replicates.

In [99]:
survey.isnull().sum()

site                      0
collection date           0
temperature_c            18
pH                        0
dissolved_oxygen_mg_L    11
conductivity_uS_cm        7
n_replicates              5
dtype: int64

Run .isnull().sum(). Which four columns have gaps, and how many each?

    - temperature_c and dissolved_oxygen_mg_L, conducitivity_uS_cm, and n_replicates are missing data.

In [100]:
survey.duplicated().sum()

20

Run .duplicated().sum(). How many rows are exact copies of an earlier row?


    - There are 20 rows that are exact copies of an earlier row.

In [101]:

survey['site'].value_counts()

site
site-b     16
 site-c    13
SITE_F     12
 site_a    12
Site_A     12
site_d     12
 site_d    12
SITE_A     12
site_f     12
site_e     12
site_f     11
site-e     11
SITE_C     11
Site_F     11
SITE_B     11
Site_C      9
site_e      9
 SITE_E     9
site-c      8
Site_D      8
site_a      8
Site_B      8
site-d      7
site_b      7
site_b      7
SITE_D      7
site_d      7
site_c      7
SITE_E      7
site_a      6
Site_E      6
site_c      5
 Site_B     4
site-a      4
 site-f     4
site-f      3
Name: count, dtype: int64

Run survey['site'].value_counts(). There are six sites. How many distinct labels does the file contain? Look carefully at the quotation marks in the output of survey['site'].unique().

    - There are 36 rows becaue they are all characterized differently.


In [102]:
survey['temperature_c'].describe()

count    302.000000
mean     -11.206623
std      173.437298
min     -999.000000
25%       16.700000
50%       18.600000
75%       21.400000
max       26.500000
Name: temperature_c, dtype: float64

#### 5. Run .describe() on temperature_c. The minimum is not a temperature any stream has ever had… What’s going on here??

    - The data seems to be incorrectley reocrded since the min value in that column is -999

### Part 2 : Clean It

In [103]:
survey = survey.drop_duplicates()

In [104]:
survey.shape

(300, 7)

#### 6. Remove the exact duplicate rows. How many rows are left?

    - There are 300 rows left

In [105]:
survey['site'] = survey['site'].str.strip()

survey['site'] = survey['site'].str.lower()

survey['site'] = survey['site'].str.replace('-', '_')

survey["site"].value_counts()



site
site_f    50
site_d    50
site_c    50
site_a    50
site_e    50
site_b    50
Name: count, dtype: int64

#### 7. Fix the site column so that all six sites have one label each. You will need three separate statements, one per method, each assigned back to survey['site']: strip the whitespace, lower-case the text, and replace the hyphens with underscores. Confirm with .value_counts() that you have exactly six distinct site labels, reading site_a through site_f

    - did it :)

In [106]:
survey['pH'] = survey['pH'].str.replace(',', '.').astype(float)


survey["pH"].value_counts()



pH
7.74    7
6.80    7
6.73    6
6.54    6
6.53    5
       ..
6.85    1
7.70    1
6.15    1
7.23    1
6.25    1
Name: count, Length: 151, dtype: int64

#### 8. pH came in as text. Find out why by looking at survey['pH'].unique(), then fix it with one .str.replace() and one .astype(), in that order. Confirm the dtype is float64 and that .describe() gives a plausible pH range.


In [107]:
survey = survey.dropna(subset = ['temperature_c', 'dissolved_oxygen_mg_L', 'conductivity_uS_cm'])

survey.shape

(264, 7)

#### 9. Three measurement columns have blanks: temperature_c, dissolved_oxygen_mg_L, and conductivity_uS_cm. A row with no measurement is no use to you, so drop those rows in a single .dropna() call with a list in subset=. How many rows did that cost?

    - We now have 264.

In [108]:

survey['n_replicates'] = survey['n_replicates'].fillna(1)

survey['n_replicates'].value_counts()

n_replicates
3.0    158
2.0     55
4.0     46
1.0      5
Name: count, dtype: int64

#### 10. n_replicates also has blanks, but here a blank means the field sheet recorded a single bottle and nobody bothered to write “1”. Fill those with 1 instead of dropping the rows, then convert the column to int. Confirm with .value_counts().

    - Confirmed

In [109]:

survey_new = survey[(survey['temperature_c'] > -100)].copy()

survey_new.shape

(255, 7)

#### 11. Now let’s deal with the impossible temperatures. Use the filter pattern from Day 3 to keep only the rows where temperature_c is above -100, and end the line with .copy(). How many rows did the loggers ruin

    - There are now 255 rows.

In [110]:
survey['temperature_c'].describe()

count    264.000000
mean     -15.493182
std      185.146146
min     -999.000000
25%       16.700000
50%       18.850000
75%       21.450000
max       26.500000
Name: temperature_c, dtype: float64

#### 12. Re-run .describe() on temperature_c. Compare the mean to the one you got in task 5. In a markdown cell, write one short statement about what the sentinel rows did to the average you saw in task 5.


    - It did not change.

In [111]:

survey = survey.rename(columns = {'collection date': 'collection_date'})

survey.head()

,site,collection_date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4.0
1,site_d,2025-08-05,12.5,7.74,9.97,300.1,4.0
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4.0
3,site_d,2025-08-15,16.0,7.74,9.73,250.7,3.0
4,site_a,2025-06-24,14.7,7.64,9.19,338.8,3.0


#### 13. Rename collection date to collection_date, so you can reach it without quoting trouble later. .rename() is from Day 2; it takes columns= and a dictionary

    - done

### Part 3. Transform It

In [112]:
survey['conductivity_mS_cm'] = survey['conductivity_uS_cm'] / 1000

survey.head()

,site,collection_date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates,conductivity_mS_cm
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4.0,0.8991
1,site_d,2025-08-05,12.5,7.74,9.97,300.1,4.0,0.3001
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4.0,0.8158
3,site_d,2025-08-15,16.0,7.74,9.73,250.7,3.0,0.2507
4,site_a,2025-06-24,14.7,7.64,9.19,338.8,3.0,0.3388


#### 14. Add a column called conductivity_mS_cm holding conductivity in millisiemens per centimetre, which is the microsiemens value divided by 1000.


    - done :)

In [113]:
def celcius_to_fahrenheit(celcius):
    fahrenheit = (celcius * 9 / 5) + 32
    return fahrenheit


survey['temperature_f'] = survey['temperature_c'].apply(celcius_to_fahrenheit)

survey.head()

,site,collection_date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates,conductivity_mS_cm,temperature_f
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4.0,0.8991,75.38
1,site_d,2025-08-05,12.5,7.74,9.97,300.1,4.0,0.3001,54.50
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4.0,0.8158,72.68
3,site_d,2025-08-15,16.0,7.74,9.73,250.7,3.0,0.2507,60.80
4,site_a,2025-06-24,14.7,7.64,9.19,338.8,3.0,0.3388,58.46


#### 15. Add a column called temperature_f holding the temperature in Fahrenheit. Do it twice: once with the derived-column pattern and plain arithmetic, and once by writing a function celsius_to_fahrenheit and using .apply(). Check that the two columns agree.



In [114]:
def classify_ph(ph):
    if ph < 6.5:
        return 'acidic'
    elif ph > 7.5:
        return 'alkaline'
    else:
        return 'neutral'

survey_new['ph_class'] = survey_new['pH'].apply(classify_ph)

survey_new['ph_class'].value_counts()

ph_class
neutral     171
acidic       42
alkaline     42
Name: count, dtype: int64

#### 16. Write a function called classify_ph that takes a pH value and returns 'acidic' below 6.5, 'alkaline' above 7.5, and 'neutral' in between. Apply it to the pH column, store the result in a column called ph_class, and report the counts.

    - neutral = 176
    - acidic = 45
    - alkaline = 43

### Part 4. Ask it Something

In [115]:
survey_new[['site', 'ph_class']]

,site,ph_class
0,site_f,acidic
1,site_d,alkaline
2,site_c,acidic
3,site_d,alkaline
4,site_a,alkaline
...,...,...
313,site_c,neutral
315,site_b,neutral
316,site_e,neutral
318,site_c,neutral


#### 17. Use the filter pattern to build a table of just the acidic samples. Which sites do they come from? Use .value_counts() on site.

    - done :)

In [117]:
survey_acidic = survey_new[survey_new['ph_class'] == 'acidic']

In [118]:
survey_acidic

,site,collection date,temperature_c,pH,dissolved_oxygen_mg_L,conductivity_uS_cm,n_replicates,ph_class
0,site_f,2025-07-28,24.1,5.89,5.51,899.1,4.0,acidic
2,site_c,2025-07-04,22.6,6.42,7.19,815.8,4.0,acidic
21,site_f,2025-08-17,24.0,6.27,5.90,895.8,3.0,acidic
28,site_f,2025-07-10,25.5,6.21,7.13,885.8,3.0,acidic
29,site_f,2025-08-13,24.4,6.21,6.13,853.7,3.0,acidic
32,site_f,2025-07-08,24.6,6.28,7.77,805.8,3.0,acidic
39,site_f,2025-06-02,25.6,6.31,5.25,844.3,4.0,acidic
43,site_c,2025-07-24,22.2,6.46,6.48,781.8,3.0,acidic
49,site_f,2025-08-11,24.4,5.78,7.48,874.9,3.0,acidic
56,site_f,2025-07-22,24.4,6.37,5.32,886.4,2.0,acidic


In [119]:
survey_acidic['site'].value_counts()

site
site_f    27
site_c    14
site_e     1
Name: count, dtype: int64

In [123]:
survey_acidic.columns.tolist()

['site',
 'collection date',
 'temperature_c',
 'pH',
 'dissolved_oxygen_mg_L',
 'conductivity_uS_cm',
 'n_replicates',
 'ph_class']

In [136]:
site_d_data = survey_acidic[survey_acidic['site'] == 'site_f']


site_d_data['dissolved_oxygen_mg_L'].mean()


6.101481481481481

In [ ]:
site_d = survey[survey['site'] == 'site_d']
print(site_d['dissolved_oxygen_mg_L'].mean())

site_f = survey[survey['site'] == 'site_f']
print(site_f['dissolved_oxygen_mg_L'].mean())

TypeError: 'site_c_data' is an invalid keyword argument for print()

#### 18.Two of the six sites account for nearly all the acidic samples. Take either one of those two, and compare it against site_d, which is not one of them: filter to each in turn and compare the mean of dissolved_oxygen_mg_L. (Two filters, two .mean() calls. Tomorrow you will learn to do all six at once.)

    - 